# Organoid Picker — Brightfield Mosaic
Two workflows:
- **Manual** — click organoid centers in napari
- **Auto** — threshold + shape filter, then review/edit in napari

Both share the same napari step and export.

## 1. Settings

In [ ]:
import sys, json
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, r'C:\Users\aifadmin\scapecontrol\repo')
import opm_acquisition

OVERVIEW_DIR = Path(r'C:\Users\aifadmin\Desktop\TEST\brightfield_overview')
POS_FILE_OUT = OVERVIEW_DIR / 'PositionList_organoids.pos'

Z_OVERRIDE_UM = None   # None = use acquisition focus Z; set a number to override

## 2. Load mosaic and pixel → stage mapping

In [ ]:
mosaic = tifffile.imread(str(OVERVIEW_DIR / 'mosaic.tif')).astype(np.float32)
print(f'Mosaic: {mosaic.shape[1]} x {mosaic.shape[0]} px')

with open(OVERVIEW_DIR / 'tile_positions.json') as f:
    meta = json.load(f)

PIXEL_SIZE_UM = meta['pixel_size_um']
CAMERA_PX     = meta['camera_px']
Z_FOCUS_UM    = meta['z_focus_um']

col_to_x = {t['col']: t['x'] for t in meta['tiles']}
row_to_y = {t['row']: t['y'] for t in meta['tiles']}
max_row  = max(row_to_y.keys())
max_col  = max(col_to_x.keys())

def px_to_stage(row_px, col_px):
    c = int(np.clip(col_px // CAMERA_PX, 0, max_col))
    r = int(np.clip(max_row - row_px // CAMERA_PX, 0, max_row))
    x = col_to_x[c] + (col_px % CAMERA_PX - CAMERA_PX / 2) * PIXEL_SIZE_UM
    y = row_to_y[r] - (row_px % CAMERA_PX - CAMERA_PX / 2) * PIXEL_SIZE_UM
    return float(x), float(y)

print(f'Grid: {max_col+1} cols x {max_row+1} rows  |  Z focus: {Z_FOCUS_UM:.1f} um')

---
## 3a. Auto-detection (skip to 3b for manual-only)
Downscales the mosaic, applies Gaussian smoothing + Otsu threshold, then filters
connected regions by size and circularity. Detected centroids are passed to napari.

In [ ]:
# --- Detection parameters (edit here) ---
DOWNSCALE       = 4      # speed-up factor; effective pixel size = PIXEL_SIZE_UM * DOWNSCALE
INVERT          = True   # True if organoids are darker than background
SMOOTHING_UM    = 50     # µm — Gaussian blur before threshold (larger = smoother blobs)
MIN_DIAMETER_UM = 80     # µm — smallest organoid to keep
MAX_DIAMETER_UM = 600    # µm — largest organoid to keep
MIN_CIRCULARITY = 0.5    # 0–1 (1 = perfect circle); raise to reject elongated debris

In [ ]:
from skimage.transform import downscale_local_mean
from skimage import filters, measure, morphology

px_eff      = PIXEL_SIZE_UM * DOWNSCALE
min_area_px = np.pi * (MIN_DIAMETER_UM / 2 / px_eff) ** 2
max_area_px = np.pi * (MAX_DIAMETER_UM / 2 / px_eff) ** 2

# Downscale for speed
small = downscale_local_mean(mosaic, (DOWNSCALE, DOWNSCALE)).astype(np.float32)

# Normalise
p2, p98 = np.percentile(small, (2, 98))
img = np.clip((small - p2) / (p98 - p2 + 1e-9), 0, 1)
if INVERT:
    img = 1 - img

# Smooth → Otsu threshold
sigma_px   = SMOOTHING_UM / px_eff
img_smooth = filters.gaussian(img, sigma=sigma_px)
thresh     = filters.threshold_otsu(img_smooth)
mask       = img_smooth > thresh

# Morphological cleanup
mask = morphology.remove_small_objects(mask, min_size=int(min_area_px))
mask = morphology.remove_small_holes(mask, area_threshold=int(min_area_px))

# Label and filter by size + circularity
labeled = measure.label(mask)
props   = measure.regionprops(labeled)

auto_points = []  # (row_px_full, col_px_full)
for p in props:
    if not (min_area_px <= p.area <= max_area_px):
        continue
    circ = 4 * np.pi * p.area / (p.perimeter ** 2) if p.perimeter > 0 else 0
    if circ < MIN_CIRCULARITY:
        continue
    # Half-pixel correction: downscale block centre → full-res centre
    row_full = p.centroid[0] * DOWNSCALE + (DOWNSCALE - 1) / 2
    col_full = p.centroid[1] * DOWNSCALE + (DOWNSCALE - 1) / 2
    auto_points.append((row_full, col_full))

print(f'Detected {len(auto_points)} organoids')

# Preview
p1, p99 = np.percentile(mosaic, (1, 99))
display  = np.clip((mosaic - p1) / (p99 - p1 + 1e-9), 0, 1)
fig, ax  = plt.subplots(figsize=(12, 12))
ax.imshow(display, cmap='gray')
if auto_points:
    pts = np.array(auto_points)
    ax.scatter(pts[:, 1], pts[:, 0], s=80, facecolors='none', edgecolors='red', linewidths=1.5)
ax.set_title(f'{len(auto_points)} detections')
ax.axis('off'); plt.tight_layout(); plt.show()

---
## 3b. Open in napari — review / edit / add points
Auto-detected points are pre-loaded (run 3a first) or start empty for manual-only.
Select the **Organoids** layer, use **+** to add, or select + Delete to remove.

In [ ]:
import napari
from magicgui import magicgui
from skimage.transform import downscale_local_mean
from skimage import filters, measure, morphology

# Pre-compute the normalised detection image once (shared with widget)
px_eff = PIXEL_SIZE_UM * DOWNSCALE
_small = downscale_local_mean(mosaic, (DOWNSCALE, DOWNSCALE)).astype(np.float32)
_p2, _p98 = np.percentile(_small, (2, 98))
_img = np.clip((_small - _p2) / (_p98 - _p2 + 1e-9), 0, 1)
if INVERT:
    _img = 1 - _img

def _detect(smoothing_um, threshold_factor, min_diam_um, max_diam_um, min_circ):
    """Threshold + regionprops detection; returns (N×2 points, N sizes) in full-res px."""
    min_area = np.pi * (min_diam_um / 2 / px_eff) ** 2
    max_area = np.pi * (max_diam_um / 2 / px_eff) ** 2

    smooth = filters.gaussian(_img, sigma=smoothing_um / px_eff)
    otsu   = filters.threshold_otsu(smooth)
    mask   = smooth > otsu * threshold_factor
    mask   = morphology.remove_small_objects(mask, min_size=int(min_area))
    mask   = morphology.remove_small_holes(mask, area_threshold=int(min_area))

    props  = measure.regionprops(measure.label(mask))
    pts, sizes = [], []
    for p in props:
        if not (min_area <= p.area <= max_area):
            continue
        circ = 4 * np.pi * p.area / (p.perimeter ** 2) if p.perimeter > 0 else 0
        if circ < min_circ:
            continue
        pts.append((p.centroid[0] * DOWNSCALE + (DOWNSCALE-1)/2,
                    p.centroid[1] * DOWNSCALE + (DOWNSCALE-1)/2))
        # diameter in full-res px from detected area
        sizes.append(2 * np.sqrt(p.area / np.pi) * DOWNSCALE)
    if not pts:
        return np.empty((0, 2)), np.array([])
    return np.array(pts), np.array(sizes)

# Initial detection (use results from cell 3a if already run, else re-detect)
if 'auto_points' in dir() and auto_points:
    init_pts   = np.array(auto_points)
    init_sizes = np.full(len(auto_points), POINT_SIZE_UM / PIXEL_SIZE_UM)
else:
    POINT_SIZE_UM = 150
    init_pts, init_sizes = _detect(SMOOTHING_UM, 1.0, MIN_DIAMETER_UM, MAX_DIAMETER_UM, MIN_CIRCULARITY)

POINT_SIZE_UM = 150   # default size for manually added points (µm diameter)

# ── Open napari ───────────────────────────────────────────────────────────────
viewer = napari.Viewer()
viewer.add_image(mosaic, name='Brightfield mosaic', colormap='gray')

points_layer = viewer.add_points(
    init_pts,
    name='Organoids',
    size=init_sizes if len(init_sizes) else POINT_SIZE_UM / PIXEL_SIZE_UM,
    face_color='transparent',
    border_color='red',
    border_width=0.05,
)
points_layer.current_size = POINT_SIZE_UM / PIXEL_SIZE_UM
points_layer.mode = 'select'

# ── Re-detection widget ───────────────────────────────────────────────────────
@magicgui(
    auto_call=False,
    call_button='Re-detect',
    smoothing   ={'widget_type': 'Slider',      'min': 10,   'max': 200,
                  'value': SMOOTHING_UM,                     'label': 'Smoothing (µm)'},
    thresh_fac  ={'widget_type': 'FloatSlider', 'min': 0.5,  'max': 2.0,
                  'step': 0.05, 'value': 1.0,               'label': 'Threshold ×Otsu'},
    min_diam    ={'widget_type': 'Slider',      'min': 20,   'max': 500,
                  'value': MIN_DIAMETER_UM,                  'label': 'Min diameter (µm)'},
    max_diam    ={'widget_type': 'Slider',      'min': 50,   'max': 2000,
                  'value': MAX_DIAMETER_UM,                  'label': 'Max diameter (µm)'},
    min_circ    ={'widget_type': 'FloatSlider', 'min': 0.0,  'max': 1.0,
                  'step': 0.05, 'value': MIN_CIRCULARITY,   'label': 'Min circularity'},
)
def redetect(smoothing=SMOOTHING_UM, thresh_fac=1.0,
             min_diam=MIN_DIAMETER_UM, max_diam=MAX_DIAMETER_UM,
             min_circ=MIN_CIRCULARITY):
    pts, szs = _detect(smoothing, thresh_fac, min_diam, max_diam, min_circ)
    points_layer.data = pts
    if len(szs):
        points_layer.size = szs
        points_layer.current_size = float(np.median(szs))
    print(f'{len(pts)} organoids  (smoothing={smoothing} µm, thresh×{thresh_fac:.2f}, circ≥{min_circ:.2f})')

viewer.window.add_dock_widget(redetect, name='Detection controls', area='right')

print(f'{len(init_pts)} initial detections.')
print()
print('  Detection panel (right):  adjust sliders → Re-detect')
print('  Threshold ×Otsu > 1.0  →  raise threshold (fewer detections)')
print('  Threshold ×Otsu < 1.0  →  lower threshold (more detections)')
print('  Select mode (active):  click → Delete to remove a point')
print('  Add mode:              press A to add missed organoids')
print()
print('Run cell 4 when done — napari can stay open.')

## 4. Export position list

In [ ]:
from opm_acquisition import XY_STAGE, Z_STAGE

picked  = points_layer.data
z_value = float(Z_OVERRIDE_UM if Z_OVERRIDE_UM is not None else Z_FOCUS_UM)
print(f'{len(picked)} positions to export')

if len(picked) == 0:
    raise RuntimeError('No points — add some in napari first.')

def make_position(label, x, y, z):
    return {
        'DefaultXYStage': {'type': 'STRING', 'scalar': XY_STAGE},
        'DefaultZStage':  {'type': 'STRING', 'scalar': Z_STAGE},
        'DevicePositions': {
            'type': 'PROPERTY_MAP',
            'array': [
                {'Device': {'type': 'STRING', 'scalar': Z_STAGE},
                 'Position_um': {'type': 'DOUBLE', 'array': [float(z)]}},
                {'Device': {'type': 'STRING', 'scalar': XY_STAGE},
                 'Position_um': {'type': 'DOUBLE', 'array': [float(x), float(y)]}},
            ],
        },
        'GridCol':    {'type': 'INTEGER',      'scalar': 0},
        'GridRow':    {'type': 'INTEGER',      'scalar': 0},
        'Label':      {'type': 'STRING',       'scalar': label},
        'Properties': {'type': 'PROPERTY_MAP', 'scalar': {}},
    }

positions = []
for i, (row_px, col_px) in enumerate(picked):
    x_um, y_um = px_to_stage(row_px, col_px)
    positions.append(make_position(f'Pos{i}', x_um, y_um, z_value))

pos_data = {
    'encoding': 'UTF-8',
    'format': 'Micro-Manager Property Map',
    'major_version': 2, 'minor_version': 0,
    'map': {'StagePositions': {'type': 'PROPERTY_MAP', 'array': positions}},
}
with open(POS_FILE_OUT, 'w') as f:
    json.dump(pos_data, f, indent=2)

print(f'Saved -> {POS_FILE_OUT}')

## 5. Preview

In [ ]:
xs = [p['DevicePositions']['array'][1]['Position_um']['array'][0] for p in positions]
ys = [p['DevicePositions']['array'][1]['Position_um']['array'][1] for p in positions]

x0 = col_to_x[0]       - CAMERA_PX / 2 * PIXEL_SIZE_UM
x1 = col_to_x[max_col] + CAMERA_PX / 2 * PIXEL_SIZE_UM
y0 = row_to_y[max_row] - CAMERA_PX / 2 * PIXEL_SIZE_UM
y1 = row_to_y[0]       + CAMERA_PX / 2 * PIXEL_SIZE_UM

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(display, cmap='gray', extent=[x0, x1, y0, y1], origin='upper')
ax.scatter(xs, ys, facecolors='none', edgecolors='red', s=120, linewidths=1.5)
for i, (x, y) in enumerate(zip(xs, ys)):
    ax.annotate(str(i), (x, y), color='yellow', fontsize=8, ha='left')
ax.set_xlabel('X (um)'); ax.set_ylabel('Y (um)')
ax.set_title(f'{len(positions)} organoids')
plt.tight_layout(); plt.show()